#### Forecast API

##### Llamada a la API Open Meteo Forecast

In [9]:
import openmeteo_requests
import requests_cache
import pandas as pd
import numpy as np
from retry_requests import retry
from pathlib import Path

def main():
    # 1) Forecast meteo
    cache = requests_cache.CachedSession('.cache', expire_after=3600)
    sess  = retry(cache, retries=5, backoff_factor=0.2)
    client = openmeteo_requests.Client(session=sess)

    vars_hr = [  # tu lista completa
        "temperature_2m", "dew_point_2m", "relative_humidity_2m", "apparent_temperature",
        "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "visibility",
        "evapotranspiration", "et0_fao_evapotranspiration", "vapour_pressure_deficit", "wind_speed_10m",
        "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m",
        "wind_direction_120m", "wind_direction_180m", "wind_gusts_10m", "temperature_80m", "temperature_120m",
        "temperature_180m", "soil_temperature_0cm", "soil_temperature_6cm", "soil_temperature_18cm",
        "soil_temperature_54cm", "soil_moisture_0_to_1cm", "soil_moisture_1_to_3cm", "soil_moisture_3_to_9cm",
        "soil_moisture_9_to_27cm", "soil_moisture_27_to_81cm", "uv_index", "uv_index_clear_sky", "is_day",
        "sunshine_duration", "wet_bulb_temperature_2m", "cape", "lifted_index", "convective_inhibition",
        "freezing_level_height", "shortwave_radiation", "diffuse_radiation",
        "global_tilted_irradiance", "shortwave_radiation_instant", "diffuse_radiation_instant",
        "global_tilted_irradiance_instant", "direct_radiation", "direct_normal_irradiance",
        "terrestrial_radiation", "direct_radiation_instant", "direct_normal_irradiance_instant",
        "terrestrial_radiation_instant", "pressure_msl"
    ]

    params = {
        "latitude": 18.2158,
        "longitude": -71.0998,
        "hourly": vars_hr,
        "timezone": "UTC",
        "past_days": 1,
        "forecast_days": 7,
        "models": "best_match"
    }

    url = "https://api.open-meteo.com/v1/forecast"
    resps = client.weather_api(url, params=params)
    if not resps:
        raise RuntimeError("No response from forecast API")
    hr = resps[0].Hourly()
    t0 = pd.to_datetime(hr.Time(),      unit="s", utc=True)
    t1 = pd.to_datetime(hr.TimeEnd(),   unit="s", utc=True)
    freq = pd.Timedelta(seconds=hr.Interval())
    idx  = pd.date_range(t0, t1, freq=freq, inclusive="left", tz="UTC")

    df_m = pd.DataFrame(
        {v: hr.Variables(i).ValuesAsNumpy() for i,v in enumerate(vars_hr)},
        index=idx
    )

    # 2) Load histórico de generación
    hist_file = Path("../data/interim/post_despacho_transformed_data/post_despacho_transformed.parquet")
    df_h = pd.read_parquet(hist_file)
    # detecta la columna de generación
    gen_cols = [c for c in df_h.columns if "gen" in c.lower()]
    if not gen_cols:
        raise KeyError(f"No generation column found in {hist_file}")
    gen_col = gen_cols[0]
    df_h = df_h[[gen_col]].rename(columns={gen_col:"generation"})
    # asegurar índice datetime UTC
    if "timestamp" in df_h.columns:
        df_h["timestamp"] = pd.to_datetime(df_h["timestamp"], utc=True)
        df_h = df_h.set_index("timestamp")
    else:
        df_h.index = pd.to_datetime(df_h.index, utc=True)

    # 3) Extraer 24h previas al forecast
    start_fcst = idx.min()
    df_hist24  = df_h.loc[start_fcst - pd.Timedelta(days=1): start_fcst - pd.Timedelta(hours=1)]

    # 4) Construir columna generation: histórico + ceros
    gen = np.zeros(len(df_m), dtype=float)
    # ubica las horas históricas en el índice
    mask = df_m.index.isin(df_hist24.index)
    gen[mask] = df_hist24.reindex(df_m.index[mask]).generation.values
    df_m["generation"] = gen

    # 5) Save combined raw
    out = Path("../data/raw/forecast_meteo_data/parque_solar_girasol_forecast_api_request.csv")
    out.parent.mkdir(parents=True, exist_ok=True)
    df_m.reset_index().rename(columns={"index":"date"}).to_csv(out, index=False)
    print("✅ Saved meteo+generation raw to:", out)

if __name__ == "__main__":
    main()


✅ Saved meteo+generation raw to: ..\data\raw\forecast_meteo_data\parque_solar_girasol_forecast_api_request.csv


#### 2. FUNCIONES AUXILIARES

In [10]:
def identify_non_stationary(df, alpha=0.05):
    non_stat = []
    for col in df.select_dtypes('number').columns:
        s = df[col].dropna()
        if len(s) < 10:
            continue
        try:
            p_adf = adfuller(s)[1]
            p_kpss = kpss(s, nlags='auto')[1]
        except:
            non_stat.append(col)
            continue
        if p_adf >= alpha or p_kpss <= alpha:
            non_stat.append(col)
    return non_stat

def compute_best_lags(df, target='generation', max_lag=24):
    vars_ = [c for c in df.select_dtypes('number').columns if c != target]
    xcorr = pd.DataFrame({
        v: [df[target].corr(df[v].shift(l)) for l in range(max_lag+1)]
        for v in vars_
    }, index=range(max_lag+1))
    return {v: int(xcorr[v].abs().idxmax()) for v in vars_}

def add_temporal_features(df):
    df = df.copy()
    idx = df.index
    df['hour']     = idx.hour
    df['hour_sin'] = np.sin(2*np.pi * df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi * df['hour']/24)
    df['dow']      = idx.dayofweek
    df['dow_sin']  = np.sin(2*np.pi * df['dow']/7)
    df['dow_cos']  = np.cos(2*np.pi * df['dow']/7)
    df['month']    = idx.month
    df['month_sin']= np.sin(2*np.pi * df['month']/12)
    df['month_cos']= np.cos(2*np.pi * df['month']/12)
    return df

#### 3. TRANSFORMER PERSONALIZADO

In [11]:
class SolarFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self,
                 target='generation',
                 max_lag=24,
                 roll_windows=None,
                 log_transform_cols=None):
        self.target = target
        self.max_lag = max_lag
        self.roll_windows = roll_windows or [3,6,24]
        self.log_transform_cols = log_transform_cols or []

    def fit(self, X, y=None):
        self.non_stat_vars_ = identify_non_stationary(X)
        self.best_lags_    = compute_best_lags(X, target=self.target, max_lag=self.max_lag)
        return self

    def transform(self, X):
        df = X.copy()

        # A) Log-transform del target
        #df[f"{self.target}_log1p"] = np.log1p(df[self.target])

        # B) Clip outliers
        clip_q = [0.001, 0.999]
        num = df.select_dtypes('number').columns.drop(
            [self.target, f"{self.target}_log1p"], errors='ignore'
        )
        lowers = df[num].quantile(clip_q[0])
        uppers = df[num].quantile(clip_q[1])
        df[num] = df[num].clip(lower=lowers, upper=uppers, axis=1)

        # 1) Diferencias
        for col in self.non_stat_vars_:
            df[f"{col}_diff1"] = df[col].diff(1)

        # 2) Lags óptimos y 24h
        for col, lag in self.best_lags_.items():
            if col == self.target: continue
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
        for col in num:
            df[f"{col}_lag24"] = df[col].shift(24)

        # 3) Rolling means
        numeric = df.select_dtypes('number').columns.drop(
            [self.target, f"{self.target}_log1p"], errors='ignore'
        )
        for w in self.roll_windows:
            for col in numeric:
                df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()

        # 4) Interacciones y ratios
        for col in ['shortwave_radiation', 'global_tilted_irradiance']:
            if col in df:
                df[f"{col}_sq"] = df[col]**2
        if {'diffuse_radiation','global_tilted_irradiance'}.issubset(df.columns):
            df['diffuse_ratio'] = df['diffuse_radiation'] / (df['global_tilted_irradiance']+1e-6)

        # 5) Temporales cíclicos
        df = add_temporal_features(df)

        # 6) Log-transform extras
        for col in self.log_transform_cols:
            if col in df:
                df[f"{col}_log1p"] = np.log1p(df[col])

        # 7) Borramos cruft: las columnas originales que ya no usamos
        drop_cols = list(self.non_stat_vars_)
        df = df.drop(columns=drop_cols, errors='ignore')

        # 8) Filtrar NaNs y resetear índice si quieres
        df = df.dropna()

        # 9) Guardar feature names para usar luego en producción
        self.feature_names_ = df.columns.tolist()
        return df

    def get_feature_names_out(self):
        return self.feature_names_


#### 4. CARGA DE DATOS

In [13]:
input_path = "../data/raw/forecast_meteo_data/parque_solar_girasol_forecast_api_request.csv"
df = pd.read_csv(input_path)

#### 5. CONFIGURACIÓN DE FEATURE ENGINEERING

In [14]:
log_cols = [
    'shortwave_radiation','diffuse_radiation',
    'global_tilted_irradiance','shortwave_radiation_instant',
    'diffuse_radiation_instant','global_tilted_irradiance_instant',
    'direct_radiation','direct_normal_irradiance',
    'wind_speed_10m','wind_gusts_10m',
    'vapour_pressure_deficit','sunshine_duration'
]

sfe = SolarFeatureEngineer(
    target='generation',
    max_lag=24,
    roll_windows=[3,6,24],
    log_transform_cols=log_cols
)

# Entrenas tu transformer con los datos históricos:
sfe.fit(df)

# Y luego generas ya solo el dataset de features listo para modelar:
df_feat = sfe.transform(df)

# Si necesitas ver el listado final de columnas:
sfe.get_feature_names_out()

C:\Users\ferna\AppData\Local\Temp\ipykernel_15288\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  p_kpss = kpss(s, nlags='auto')[1]
C:\Users\ferna\AppData\Local\Temp\ipykernel_15288\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  p_kpss = kpss(s, nlags='auto')[1]
C:\Users\ferna\AppData\Local\Temp\ipykernel_15288\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  p_kpss = kpss(s, nlags='auto')[1]
C:\Users\ferna\AppData\Local\Temp\ipykernel_15288\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value

ValueError: cannot convert float NaN to integer

#### 6. GENERACIÓN DEL DATASET DE FEATURES

In [ ]:
df_feat = sfe.transform(df)

C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\2176273940.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()
C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\2176273940.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()
C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\2176273940.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fr